<a href="https://colab.research.google.com/github/tuankhoin/CO3133-Deep-Learning/blob/main/Week_6_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ho Chi Minh University of Technology (HCMUT)

CO3133 - Deep Learning and Its Applications

# Week 6 - Deep Learning for Image Segmentation

<img src="https://raw.githubusercontent.com/tuankhoin/CO3057-Computer-Vision/refs/heads/main/assets/illusion.jpg" height=400/>

### From Image Classification to Dense Prediction

In previous weeks, neural networks usually produced one output for an image or sequence.

Segmentation changes the question:

> Instead of asking **"What is in this image?"**, ask **"What is at every pixel?"**

This lecture develops the complete segmentation pipeline:

- task definition;
- UNet and encoder-decoder design;
- losses;
- metrics;
- data pipeline;
- modern segmentation models.

## Learning Outcomes

By the end of this lecture, you should be able to:

- Explain semantic, instance, and panoptic segmentation.
- Explain why dense prediction needs spatial reconstruction.
- Describe encoder, bottleneck, decoder, skip connection, neck, and head.
- Track tensor shapes through a UNet-style model.
- Choose suitable output activations and label encodings.
- Explain Cross-Entropy, Dice, IoU-style, weighted, Focal, and Tversky losses.
- Compute Dice and IoU from small masks.
- Choose metrics for region, boundary, and instance quality.
- Build a correct image-mask data pipeline without leakage.
- Explain DeepLab, SegFormer, Mask2Former, and SAM at a conceptual level.

## Lecture Contents

1. Segmentation as Dense Prediction
2. Segmentation Task Types
3. Segmentation Architecture and UNet
4. Output Representation
5. Loss Functions
6. Metrics and Evaluation
7. Data Pipeline and Loader
8. Advanced Segmentation Models
9. Practical Project Recipe

---
# 1. Segmentation as Dense Prediction
---

## [Try this demo!](https://huggingface.co/spaces/Xenova/segment-anything-web)

## Link to Week 3: CNN Classification → Segmentation

A classification CNN typically does:

$$
\text{Image}
\rightarrow
\text{Encoder}
\rightarrow
\text{Global Pooling}
\rightarrow
\text{Vector}
\rightarrow
\text{Class}.
$$

Global pooling deliberately removes **where** features occur.

For segmentation, we need:

$$
\text{Image}
\rightarrow
\text{Features}
\rightarrow
\text{Spatial Reconstruction}
\rightarrow
\text{Class at Every Pixel}.
$$

So segmentation is a **dense prediction** problem.

## What Is Image Segmentation?

<img src="https://manipulation.csail.mit.edu/data/coco_instance_segmentation.jpeg" height=300/>

Image segmentation assigns a class label to every pixel.

For an input batch:

$$
X\in\mathbb{R}^{B\times C\times H\times W},
$$

a semantic segmentation model may produce:

$$
Z\in\mathbb{R}^{B\times K\times H\times W},
$$

where $K$ is the number of classes.

Unlike classification, the output remains aligned with the image grid.

## Classification vs Detection vs Segmentation

| Task | Output | Spatial structure | Typical metric |
|---|---|---|---|
| Classification | one label / image | low | Accuracy |
| Detection | boxes + classes | medium | mAP |
| Segmentation | mask / masks | high | mIoU, Dice, PQ |

### Key idea

Segmentation combines:

1. **appearance** — what class does this region look like?
2. **location** — where exactly is it?

This is why boundaries and small structures matter.

## Why Segmentation Matters

<img src="https://cdn.prod.website-files.com/62cd5ce03261cb3e98188470/688b51c56d6c5f9871b09bf8_AD_4nXdFCCzuF2AHq4T7Pl7GLeJlPKJICpTjNiaGmJYFbhoV1jT8RzSFu4bXboWVF8jTIlVGXCaxyJwBZoPXyA3nfuB_Z2v5JAGFOrCbH4Rkfe32hTu1uip1Jelbd8B-u6OKW5Wwb2xelA.png" height=200/>

Applications include:

- **Medical:** organ, lesion, and tumor boundaries.
- **Driving / robotics:** free space, lanes, obstacles.
- **Satellite:** land-cover regions and parcels.
- **Industry:** defects and material regions.

<img src="https://raw.githubusercontent.com/tuankhoin/CO3057-Computer-Vision/refs/heads/main/assets/robotics.gif" height=300/>

A model can predict the correct class overall but still fail because:

- boundaries are inaccurate;
- thin structures disappear;
- small regions are missed.

---
# 2. Segmentation Task Types
---

## Semantic Segmentation

Semantic segmentation assigns one category to every pixel.

For one image:

$$\boxed{
Y\in\{1,\ldots,K\}^{H\times W}
}$$

Pixels with the same class receive the same semantic label, even if they belong to different physical objects.

Examples:

- road / sidewalk / car;
- tissue classes;
- land-cover classes.

It answers:

> **"What class is this pixel?"**

but not:

> **"Which particular object owns this pixel?"**

## Instance Segmentation

<img src="https://mediacy.com/wp-content/uploads/2025/08/Semantic-Segmentation.png" height=300/>

Instance segmentation separates different objects even if they share the same class.

Example:

- three people are all class `person`;
- the model must still output three different masks.

Typical output:

$$\boxed{
\{
(\text{mask}_j,\text{class}_j,\text{score}_j)
\}_{j=1}^{N}}
$$

It introduces detection-like concerns:

- grouping;
- duplicate predictions;
- matching predictions to ground truth.

## Panoptic Segmentation

<img src="https://miro.medium.com/v2/resize:fit:1400/0*qtZZQJPo6kyZ6u8o" height=300/>

Panoptic segmentation combines:

- **things** — countable objects such as people and cars;
- **stuff** — amorphous regions such as sky or road.

The goal is a complete non-overlapping partition of the image.

A common metric is:

$$
PQ=SQ\times RQ.
$$

- $SQ$: quality of matched masks.
- $RQ$: recognition/matching quality.

## Panoptic Quality

Segmentation Quality:

$$\boxed{
SQ=
\frac{
\sum_{(p,g)\in TP} IoU(p,g)
}{
|TP|
}}
$$

Recognition Quality:

$$\boxed{
RQ=
\frac{|TP|}
{|TP|+\frac12|FP|+\frac12|FN|}}
$$

Then:

$$\boxed{
PQ=SQ\times RQ}
$$

Interpretation:

- bad masks → $SQ$ decreases;
- missing/duplicate/wrong objects → $RQ$ decreases.

## Which Task Am I Solving?

| Question | Task |
|---|---|
| What class is each pixel? | Semantic segmentation |
| Which individual object owns each pixel? | Instance segmentation |
| Can I combine instance "things" and semantic "stuff"? | Panoptic segmentation |

This decision affects:

- labels;
- architecture;
- loss;
- evaluation metric.

---
# 3. Segmentation Architecture and UNet
---

## 3.1 A General Dense-Prediction Pipeline

A useful architecture view is:

$$
X
\rightarrow
\text{Backbone}
\rightarrow
\text{Neck (optional)}
\rightarrow
\text{Head}
\rightarrow
Z.
$$

For semantic segmentation:

$$
Z\in\mathbb{R}^{B\times K\times H\times W}.
$$

- **Backbone:** feature extraction and spatial reconstruction.
- **Neck:** optional refinement/fusion/context.
- **Head:** converts features into task outputs.

## Classification Backbone vs Segmentation Backbone

### Classification: $
\text{image}
\rightarrow
\text{encoder}
\rightarrow
\text{pooled vector}
\rightarrow
\text{FC}.
$

### Segmentation: $
\text{image}
\rightarrow
\text{encoder}
\rightarrow
\text{bottleneck}
\rightarrow
\text{decoder}
\rightarrow
\text{dense map}.
$

Global pooling is useful for classification because location is unnecessary.

Segmentation instead needs a path that preserves or reconstructs spatial detail.

## 3.2 UNet: The Core Mental Model

<img src="https://cms.aivietnam.edu.vn/uploads/figure_18_7cd288de78.png" height=300/>

A UNet-style backbone contains:

$$
\boxed{
\text{Encoder}
+
\text{Bottleneck}
+
\text{Decoder}
+
\text{Skip Connections}
}
$$

Conceptually:

$$\boxed{
\text{Input}
\rightarrow
\text{Encoder}
\rightarrow
\text{Bottleneck}
\rightarrow
\text{Decoder}
\rightarrow
\text{Head}}
$$

Skip connections route high-resolution encoder features directly into decoder stages.

## 3.3 Encoder: Learn Context While Reducing Resolution

Typical encoder stages use repeated:
$
\text{Conv}
\rightarrow
\text{Norm}
\rightarrow
\text{Activation}.
$

Pooling or strided convolution then reduces:

$$
(H,W)
\rightarrow
\left(\frac{H}{2},\frac{W}{2}\right).
$$

Going deeper:

- spatial resolution decreases;
- receptive field increases;
- features become more semantic.

The encoder forms a hierarchy:
$
\text{fine}
\rightarrow
\text{coarse}.
$

## 3.4 Bottleneck

The bottleneck is the deepest feature map.

Typical properties:

- smallest $H,W$;
- strongest context;
- large receptive field;
- often many channels.

It is good at understanding **what** is present, but it contains less precise spatial detail.

⟹ motivates the decoder and skip connections.

## 3.5 Decoder: Reconstruct Spatial Resolution

The decoder increases feature-map resolution toward the input size.

Common methods:

- transposed convolution;
- interpolation + convolution.

Example:

$$
(H/8,W/8)
\rightarrow
(H/4,W/4)
\rightarrow
(H/2,W/2)
\rightarrow
(H,W).
$$

Upsampling alone cannot recover spatial information already lost in the encoder.

## 3.6 Skip/Residual Connections

Downsampling can remove:

- fine boundaries;
- thin structures;
- small objects;
- high-frequency localization cues.

UNet therefore passes high-resolution encoder features directly to the decoder.

The decoder receives both:

- **deep context** — what is this region?
- **fine localization** — where exactly is the boundary?

This is the central engineering idea behind UNet.

## 3.7 Context + Localization

A useful mental model is:

$$
\boxed{
\text{Good segmentation}
=
\text{semantic context}
+
\text{precise localization}
}
$$

- Encoder/bottleneck → context.
- Skip connections → localization.
- Decoder → combines both and reconstructs a dense output.

This is a strong inductive bias for grid-structured prediction.

## 3.8 Skip Connection Fusion Methods: Concatenation vs Addition

<img src="https://www.researchgate.net/profile/Yifan-Yang-71/publication/329115979/figure/fig1/AS:695610491543552@1542857873203/Additive-skip-connections-vs-concatenative-skip-connections-Rectangles-represent-data.png" height=300/>

Suppose decoder feature $D$ and encoder skip $E$ have the same spatial size.

### Concatenation

$$\boxed{
F=\operatorname{Concat}(D,E)}
$$

- keeps feature sources distinct;
- increases channels;
- common in classical UNet.

### Addition

$$\boxed{
F=D+E}
$$

- requires matching channels;
- cheaper;
- mixes both sources immediately.

For boundary-sensitive tasks, concatenation is often the safer default.

## 3.9 Backbone, Neck, Decoder, and Head

<img src="https://www.researchgate.net/publication/344622408/figure/fig2/AS:946024113778688@1602561133932/Hybrid-architecture-for-classification-and-segmentation-with-U-Net-style-Ronneberger-et.png" height=400/>

- Decoder: Reconstructs spatial feature maps.

- Neck: Optional feature refinement:
$
F_{\text{neck}}
=
g_\phi(F_{\text{backbone}})
$
- Possible necks:
  - convolutional refinement;
  - ASPP;
  - attention;
  - FPN/BiFPN-style fusion;
  - boundary refinement.
- Head: Projects features into task outputs:
$
Z=Conv_{1\times1}(F)
$

> The decoder is not the head.

## 3.10 Segmentation Head

If final feature map is:

$$
F\in\mathbb{R}^{B\times C\times H\times W},
$$

a common semantic segmentation head is:

$$
Head(F)=Conv2d(C,K,1)(F).
$$

Output:

$$
Z\in\mathbb{R}^{B\times K\times H\times W}.
$$

The head is usually small but determines the output semantics.

## 3.11 Tensor Shapes Through a UNet

| Stage | Typical shape |
|---|---|
| Input | $(B,3,H,W)$ |
| Encoder level 1 | $(B,c_1,H,W)$ |
| Encoder level 2 | $(B,c_2,H/2,W/2)$ |
| Encoder level 3 | $(B,c_3,H/4,W/4)$ |
| Bottleneck | $(B,c_L,H_L,W_L)$ |
| Decoder | progressively returns toward $(H,W)$ |
| Logits | $(B,K,H,W)$ |

Channels encode features; $H,W$ preserve spatial layout.

## 3.12 Output-Size Mismatch

Sometimes:

$$
Z\in\mathbb{R}^{B\times K\times H'\times W'}
$$

while target mask is:

$$
Y\in\mathbb{N}^{B\times H\times W}.
$$

For pixel-wise Cross-Entropy, spatial sizes must match.

You may resize logits/features using bilinear interpolation.

But for class-index masks:

> Use **nearest-neighbor** interpolation.

Bilinear interpolation can create invalid class IDs.

## 3.13 Memory Footprint

Segmentation is memory-intensive because large spatial feature maps must be preserved.

VRAM grows with:

- batch size;
- image resolution;
- skip channels;
- decoder width.

Common responses:

- crop/patch training;
- mixed precision;
- smaller base channels;
- gradient accumulation/checkpointing.

Segmentation often reaches memory limits before accuracy limits.

## 3.14 UNet Variants

| Variant | Main change |
|---|---|
| UNet++ | nested skip pathways |
| Attention UNet | gates skip features |
| Residual UNet | residual blocks |
| ResUNet | residual/pretrained encoder + UNet decoder |

Different names often modify the same functional parts:

- encoder;
- decoder;
- skip/fusion strategy;
- neck;
- head.

## 3.15 CNN vs Transformer Segmentation

<img src="https://www.researchgate.net/publication/356919776/figure/fig1/AS:1099511569096709@1639155394318/Overview-of-Cross-Teaching-between-CNN-and-Transformer.png" height=300/>

### CNN UNet
- strong local priors;
- efficient at moderate resolution;
- natural multi-scale hierarchy.

### Transformer-based
- strong long-range mixing;
- often higher memory at high resolution.

### Hybrid
A common design is:

$$
\text{Transformer Encoder}
+
\text{CNN/UNet Decoder}.
$$

This directly links Week 5 Transformers to dense prediction.

## 3.16 How to Read Any Segmentation Paper

1. What is the encoder/backbone?
2. How is spatial resolution reduced?
3. How is it recovered?
4. Which skip or multi-scale features are fused?
5. Is there a neck?
6. What does the head output?
7. Which loss is optimized?
8. Which metric defines success?

Most segmentation papers modify one or more of these choices.

---
# 4. Output Representation
---

## Logits and Probabilities

For pixel $i$ and class $k$, the model outputs logit:
$\boxed{
z_{i,k}}
$

Softmax converts logits to class probabilities:
$\boxed{
p_{i,k}
=
\frac{
\exp(z_{i,k})
}{
\sum_{c=1}^{K}\exp(z_{i,c})
}}
$

Hard prediction:
$\boxed{
\hat y_i
=
\arg\max_k p_{i,k}}$

Many training losses expect **raw logits**, not already-softmaxed probabilities.

## Binary vs Multiclass Segmentation

### Multiclass
Exactly one of $K$ classes per pixel.

- Typical logits:
$
(B,K,H,W)
$
with softmax across classes.

- Target:
$
(B,H,W)
$
containing integer class indices.

### Binary / independent masks
- Use sigmoid outputs when channels are independent.

- The activation and loss must match the label encoding.

## Mask Encoding

### Class-index map

$$
Y\in\mathbb{N}^{H\times W}
$$

Common with `CrossEntropyLoss`.

### One-hot masks

$$
Y\in\{0,1\}^{K\times H\times W}.
$$

Useful for some overlap losses, but more memory intensive.

---
# 5. Loss Functions
---

## Why Segmentation Loss Is More Than One Softmax CE

Pixel-wise Cross-Entropy treats pixels independently.

Segmentation also faces:

- strong class imbalance;
- ambiguous boundaries;
- region-overlap objectives.

So practical systems often combine:

1. a pixel-classification term;
2. an overlap-oriented term.

## 5.1 Pixel Cross-Entropy

For valid pixel set $\mathcal{I}$:

$$\boxed{
\mathcal{L}_{CE}
=
-
\frac{1}{|\mathcal{I}|}
\sum_{i\in\mathcal{I}}
\log p_{i,y_i}}
$$

Advantages:

- stable baseline;
- works with class-index masks;
- supports `ignore_index`.

For binary or independent masks, a common alternative is `BCEWithLogitsLoss`.

## 5.2 Dice Coefficient and Dice Loss

<img src="https://www.researchgate.net/publication/379446587/figure/fig9/AS:11431281473171991@1748527298672/Intersection-over-union-IoU-Dice-and-Dice-loss.png" height=300/>

For class $k$:

$$\boxed{
Dice_k
=
\frac{
2\sum_i p_{i,k}g_{i,k}+\epsilon
}{
\sum_i p_{i,k}
+
\sum_i g_{i,k}
+
\epsilon
}}
$$

Higher Dice means stronger overlap.

A common loss is:

$$\boxed{
\mathcal{L}_{Dice}
=
1-Dice}
$$

Dice is often useful when foreground regions are small.

## 5.3 IoU / Jaccard

For class $k$:

$$\boxed{
IoU_k
=
\frac{TP_k}
{TP_k+FP_k+FN_k}}
$$

A simple IoU-style loss is:

$$\boxed{
\mathcal{L}_{IoU}
=
1-IoU}
$$

For binary sets:

$$\boxed{
Dice
=
\frac{2IoU}{1+IoU}}
$$

## Worked Example: Small Masks

Ground truth:

$$
G=
\begin{bmatrix}
1&1&0\\
1&0&0\\
0&0&0
\end{bmatrix}
$$

Prediction:

$$
\hat Y=
\begin{bmatrix}
1&0&0\\
1&1&0\\
0&0&0
\end{bmatrix}.
$$

Count foreground pixels:

- $TP=2$
- $FP=1$
- $FN=1$

Exam strategy: count $TP,FP,FN$ first.

## Worked Example: Dice and IoU

$$
Dice
=
\frac{2TP}{2TP+FP+FN}
=
\frac{4}{6}
\approx0.667.
$$

$$
IoU
=
\frac{TP}{TP+FP+FN}
=
\frac{2}{4}
=
0.5.
$$

Thus:

$$
\mathcal{L}_{Dice}=1-0.667=0.333,
$$

$$
\mathcal{L}_{IoU}=1-0.5=0.5.
$$

## Hybrid Loss: CE + Dice

A common objective is:

$$
\mathcal{L}
=
\lambda_{CE}\mathcal{L}_{CE}
+
\lambda_{Dice}\mathcal{L}_{Dice}.
$$

Intuition:

- CE gives stable per-pixel learning.
- Dice directly encourages region overlap.

Tune $\lambda$ using validation performance, not training loss alone.

## 5.3 Handling Imbalance

### Weighted Cross-Entropy
Increase penalty on rare classes.

### Focal Loss
Emphasizes difficult pixels and downweights easy background pixels.

### Tversky Loss
Allows asymmetric weighting of false positives and false negatives.

Useful when the cost of missing a region differs from the cost of a false alarm.

## 5.4 Choosing a Loss

A sensible workflow:

1. Start with CE + correct `ignore_index`.
2. Verify image-mask alignment and augmentation.
3. Inspect per-class metrics.
4. Add Dice/IoU if region overlap is weak.
5. Add class weighting/Focal/Tversky only if imbalance or error costs justify it.
6. Tune on validation metrics.

Do not make the loss complicated before verifying the dataset.

---
# 6. Metrics and Evaluation
---

## Loss vs Metric

A **loss** is optimized during training.

A **metric** defines how performance is judged.

Example:

- train with CE + Dice loss;
- report mIoU and Dice;
- add boundary metrics when edge quality matters.

The metric should reflect deployment risk.

## IoU and Mean IoU

Per-class:

$$\boxed{
IoU_k
=
\frac{TP_k}
{TP_k+FP_k+FN_k}}
$$

Mean IoU:

$$
mIoU
=
\frac{1}{K}
\sum_{k=1}^{K}
IoU_k.
$$

mIoU is a macro average: every class contributes equally.

## Dice Metric

For class $k$:

$$\boxed{
Dice_k
=
\frac{
2TP_k
}{
2TP_k+FP_k+FN_k
}}
$$

Dice is widely reported in medical segmentation.

Like IoU, it measures overlap rather than simply counting correct pixels.

## Pixel Accuracy Can Be Misleading


<img src="https://b.thumbs.redditmedia.com/EF6tx2zl9JUecJvPi73e4ywK3zcjnnuosOKA2tCyiRs.jpg" height=300/>

Pixel accuracy:

$$\boxed{
PixAcc
=
\frac{
\sum_k TP_k
}{
\sum_k(TP_k+FN_k)
}}
$$

If 95% of an image is background, predicting background everywhere may look accurate.

Always inspect:

- per-class IoU;
- per-class Dice;
- class frequency.

## Boundary Metric: HD95

<img src="https://www.researchgate.net/publication/365454013/figure/fig2/AS:11431281098016225@1668744532091/An-illustration-of-how-the-Hausdorff-95-distance-Hd95-is-computed-between-two-example.png" height=200/>

HD95 measures disagreement between predicted and ground-truth contours using a robust 95th-percentile distance.

Useful when boundaries are critical:

- surgical margins;
- thin structures;
- edge-sensitive applications.

Interpretation:

> Lower HD95 is better.

## Choosing Metrics by Failure Cost

| Concern | Useful metric |
|---|---|
| Region overlap | IoU / mIoU |
| Medical overlap | Dice |
| Boundary quality | HD95 / boundary-F |
| Instance masks | AP family |
| Panoptic output | PQ |
| Severe imbalance | per-class IoU/Dice + macro metrics |

Choose metrics based on the errors that matter.

## Evaluation Protocol Matters

For valid comparison:

- logits and masks must share the same coordinate grid;
- resize/crop operations must stay aligned;
- `ignore_index` must be applied consistently;
- decide whether void/background is included;
- select checkpoints using validation, not test.

A preprocessing mismatch can silently corrupt Dice and IoU.

---
# 7. Data Pipeline and Loader
---

## Image-Mask Pairing

A semantic segmentation dataset usually has one mask per image.

Example:

```text
images/
    001.png
    002.png

masks/
    001.png
    002.png
```

or a CSV/JSON explicitly maps image and mask paths.

Correct pairing is fundamental.

## Tensor Contract After Loading

Image:

$$
X\in\mathbb{R}^{3\times H\times W}
$$

or batched:

$$
X\in\mathbb{R}^{B\times3\times H\times W}.
$$

For Cross-Entropy, mask:

$$
Y\in\mathbb{N}^{H\times W}
$$

or:

$$
Y\in\mathbb{N}^{B\times H\times W}.
$$

The mask stores integer class IDs.

## Augmentation Rule: Geometry Must Match

If the image is:

- cropped;
- flipped;
- rotated;
- resized;

the same geometric transform must be applied to the mask.

Otherwise the labels no longer correspond to the image.

For class-index masks:

> Use **nearest-neighbor interpolation**.

## Split Discipline and Leakage

Split by the correct independent unit.

Examples:

- medical → patient;
- video → video/scene;
- repeated captures → source/session.

Avoid:

- augmented versions across different splits;
- near-duplicate frames across train/validation;
- test-set tuning.

Leakage can produce fake high mIoU/Dice.

## Visual Sanity Checks

Before training, inspect image-mask overlays.

Check:

1. image-mask alignment;
2. class-ID mapping;
3. colormap interpretation;
4. nearest-neighbor mask resize;
5. ignore-label handling;
6. augmentation alignment;
7. one complete mini-batch.

Many "model not learning" problems are really data-pipeline bugs.

## Minimal PyTorch Dataset Pattern

```python
class SegDataset(torch.utils.data.Dataset):
    def __getitem__(self, idx):
        img = load_image(self.paths[idx])
        msk = load_mask(self.mask_paths[idx])

        img, msk = self.transforms(img, msk)

        return img, msk.long()
```

For multiclass CE:

- image → floating-point tensor;
- mask → integer `long` tensor.

## DataLoader

```python
loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)
```

If image sizes vary, either:

- resize/crop to common dimensions;
- or use a custom `collate_fn`.

Batching choices affect speed and memory.

## Ignore Index and Class Frequency

Some pixels may be:

- unknown;
- void;
- unlabeled.

They should not contribute to loss or metrics.

Use a consistent `ignore_index`.

Before choosing class weights, inspect class frequency.

Common bugs:

- label IDs off by one;
- wrong void ID;
- absent classes;
- overwhelming background.

---
# 8. Advanced Segmentation Models
---

UNet is the core CNN mental model, but modern systems modify context, fusion, or mask prediction.

## DeepLab and ASPP

<img src="https://2.bp.blogspot.com/-gxnbZ9w2Dro/WqMOQTJ_zzI/AAAAAAAACeA/dyLgkY5TnFEf2j6jyXDXIDWj_wrbHhteQCLcBGAs/s1600/image2.png" height=300/>

- DeepLab uses atrous/dilated convolution to expand receptive field without reducing resolution too aggressively.

- ASPP combines several dilation rates to capture context at multiple scales.

Useful when strong semantic context is needed for scene parsing.

> This connects directly to dilated convolution from Week 3.

## SegFormer

<img src="https://camo.githubusercontent.com/e35ce478635bdf6bd9fb1cc823ebc9a12d459067076e519365a07247c04a843b/68747470733a2f2f68756767696e67666163652e636f2f64617461736574732f68756767696e67666163652f646f63756d656e746174696f6e2d696d616765732f7265736f6c76652f6d61696e2f736567666f726d65725f6172636869746563747572652e706e67" height=300/>

SegFormer combines:

- hierarchical Transformer encoder;
- lightweight MLP fusion/decoder.

Why useful:

- long-range Transformer mixing;
- hierarchical multi-scale features;
- relatively lightweight decoder.

This directly connects Week 5 Transformers to dense visual prediction.

## Mask2Former

<img src="https://debuggercafe.com/wp-content/uploads/2024/01/mask2former-architecture.png" height=300/>

Mask2Former uses Transformer decoder queries for **mask classification**.

Rather than only asking every pixel for a class, it predicts a set of mask-oriented regions.

A major advantage:

> One framework can support semantic, instance, and panoptic segmentation.

## SAM - Segment Anything

<img src="https://cdn.learnopencv.com/wp-content/uploads/2023/04/04090621/segment-anything-model.png" height=300/>

SAM is a promptable segmentation foundation model.

Prompts can include:

- points;
- boxes;
- masks.

Typical uses:

- interactive segmentation;
- annotation support;
- rapid prototyping.

A pretrained foundation model may still require adaptation under domain shift.

## Weak and Semi-Supervised Segmentation

Pixel-level annotation is expensive.

### Weak supervision
Use cheaper labels:

- boxes;
- scribbles;
- image tags;
- pseudo-labels.

### Semi-supervised learning
Use a small labeled set and many unlabeled images.

Common ideas:

- consistency regularization;
- confidence-based pseudo-label filtering.

Trade-off:

$$
\text{less labeling}
\leftrightarrow
\text{more algorithmic complexity/noise}.
$$

## Foundation-Model Direction

Large pretrained visual models can act as:

- feature extractors;
- teachers;
- promptable annotation tools;
- shared backbones;
- sources for adapters.

The largest model is not always the best runtime model.

A large model may help development while a smaller model is deployed.

---
# 9. Practical Project Recipe
---


<img src="https://i.imgflip.com/94ze4f.jpg" height=300/>

## A Practical Segmentation Workflow

### 1. Freeze the task
Choose semantic / instance / panoptic and define the metric.

### 2. Build the dataset contract
Verify class IDs, ignore rules, splits, and visualization.

### 3. Start simple
Use:

$$
\text{UNet}+\text{Cross-Entropy}.
$$

### 4. Improve based on evidence
Use per-class metrics and qualitative results to decide the next change.

## Improve Based on Failure Mode

- Poor overlap / small foreground: Add Dice/IoU-style loss.
- Strong class imbalance: Try weighted CE, Focal, or sampling changes.
- Poor boundaries: Improve resolution, skips, decoder, or boundary-aware objectives.
- Insufficient context:
  - larger receptive field;
  - ASPP;
  - stronger encoder;
  - Transformer/hybrid backbone.

Architecture changes should respond to observed errors.

## Common Mistakes

- Bilinear-resizing class-index masks.
- Inconsistent `ignore_index`.
- Image-mask misalignment.
- Label IDs off by one.
- Train/validation leakage.
- Tuning on the test set.
- Reporting only pixel accuracy under imbalance.
- Applying softmax before a loss that expects logits.
- Using complicated losses before validating the dataset.
- Ignoring per-class results.

## Different Views on Segmentation Systems

### Basic UNet

$$
\text{Input}
\rightarrow
\text{UNet Backbone}
\rightarrow
\text{Head}.
$$

### Practical modern segmentation

$$
\text{Input}
\rightarrow
\text{Visual Encoder}
\rightarrow
\text{Decoder}
\rightarrow
\text{Neck(optional)}
\rightarrow
\text{Head}.
$$

### Research view

$$
\text{Input}
\rightarrow
\text{Backbone}
\rightarrow
\text{Neck(s)}
\rightarrow
\text{Task Head(s)}
\rightarrow
\text{Losses/Metrics}.
$$

The names matter less than the **function of each component**.

---
# Week 6 Summary
---

Segmentation is structured dense prediction:

$$
\boxed{
\text{Image}
\rightarrow
\text{Encoder}
\rightarrow
\text{Context}
\rightarrow
\text{Spatial Reconstruction}
\rightarrow
\text{Per-pixel / per-mask Prediction}
}
$$

Main ideas:

- segmentation must preserve semantics **and** location;
- UNet combines deep context with high-resolution skip features;
- label encoding determines activation and loss;
- overlap losses complement pixel-wise Cross-Entropy;
- mIoU/Dice are often more informative than pixel accuracy;
- data alignment and interpolation rules are first-class engineering concerns;
- modern models change the backbone, fusion, decoder, or mask formulation, while solving the same dense-prediction problem.

## Week 6 Checklist

You should now be able to answer:

1. Why is segmentation called dense prediction?
2. How do semantic, instance, and panoptic segmentation differ?
3. Why is global pooling unsuitable for segmentation?
4. What do encoder, bottleneck, decoder, neck, and head do?
5. Why are UNet skip connections important?
6. Concatenation vs addition: what changes?
7. What are the expected logits and mask shapes?
8. When do we use softmax vs sigmoid?
9. Why must class-index masks use nearest-neighbor resizing?
10. Write and interpret Dice and IoU.
11. Why combine CE and Dice?
12. Why can pixel accuracy be misleading?
13. When is HD95 useful?
14. How can data leakage occur in segmentation?
15. Conceptually, how do DeepLab, SegFormer, Mask2Former, and SAM differ?
16. What is a sensible baseline workflow for a new segmentation project?